# Explorotry Analysis

## Dataset overview

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.home() / "Desktop"
CSV_PATH = BASE_DIR / "processed" / "buildings_all_with_crops.csv"

OUTPUT_DIR = BASE_DIR / "processed" / "initial_dataset_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)

print(f"Loaded rows: {len(df):,}")
print("\nColumns:")
print(df.columns.tolist())

overview = {
    "total_building_instances": len(df),
    "num_columns": len(df.columns),
    "num_splits": df["split"].nunique() if "split" in df.columns else None,
    "num_disasters": df["disaster"].nunique() if "disaster" in df.columns else None,
    "num_images": df["image_id"].nunique() if "image_id" in df.columns else None,
    "num_crop_paths": df["crop_path"].nunique() if "crop_path" in df.columns else None,
}

overview_df = pd.DataFrame([overview])

print("\nDataset overview:")
print(overview_df.to_string(index=False))

overview_df.to_csv(OUTPUT_DIR / "01_dataset_overview.csv", index=False)

if "split" in df.columns:
    split_counts = df["split"].value_counts().reset_index()
    split_counts.columns = ["split", "count"]
    split_counts.to_csv(OUTPUT_DIR / "01_split_counts.csv", index=False)
    print("\nSplit counts:")
    print(split_counts.to_string(index=False))

if "damage_label" in df.columns:
    label_counts = df["damage_label"].value_counts().reset_index()
    label_counts.columns = ["damage_label", "count"]
    label_counts.to_csv(OUTPUT_DIR / "01_damage_label_counts.csv", index=False)
    print("\nDamage label counts:")
    print(label_counts.to_string(index=False))

if "disaster" in df.columns:
    disaster_counts = df["disaster"].value_counts().reset_index()
    disaster_counts.columns = ["disaster", "count"]
    disaster_counts.to_csv(OUTPUT_DIR / "01_disaster_counts.csv", index=False)
    print("\nDisaster counts:")
    print(disaster_counts.to_string(index=False))

print("\nSaved outputs to:")
print(OUTPUT_DIR)

Loaded rows: 266,780

Columns:
['instance_id', 'split', 'disaster', 'image_id', 'pre_image_path', 'post_image_path', 'label_json_path', 'building_index', 'building_uuid', 'damage_label', 'wkt', 'xmin', 'ymin', 'xmax', 'ymax', 'bbox_width', 'bbox_height', 'crop_path']

Dataset overview:
 total_building_instances  num_columns  num_splits  num_disasters  num_images  num_crop_paths
                   266780           18           3             10        3731          266780

Split counts:
split  count
train 159793
 test  53850
 hold  53137

Damage label counts:
damage_label  count
   no-damage 196823
minor-damage  26116
major-damage  22616
   destroyed  21225

Disaster counts:
           disaster  count
       palu-tsunami  55178
  mexico-earthquake  51362
   hurricane-harvey  37374
  hurricane-michael  35127
  hurricane-matthew  22630
santa-rosa-wildfire  21869
         socal-fire  18276
   midwest-flooding  13423
 hurricane-florence  10728
  guatemala-volcano    813

Saved outputs to:
/U

## Class distribution

In [2]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.home() / "Desktop"
CSV_PATH = BASE_DIR / "processed" / "buildings_all_with_crops.csv"

OUTPUT_DIR = BASE_DIR / "processed" / "initial_dataset_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_ORDER = [
    "no-damage",
    "minor-damage",
    "major-damage",
    "destroyed",
]

df = pd.read_csv(CSV_PATH)

df = df[df["damage_label"].isin(LABEL_ORDER)].copy()

global_counts = (
    df["damage_label"]
    .value_counts()
    .reindex(LABEL_ORDER)
    .fillna(0)
    .astype(int)
    .reset_index()
)

global_counts.columns = ["damage_label", "count"]
global_counts["percentage"] = global_counts["count"] / global_counts["count"].sum() * 100

print("\nGlobal class distribution:")
print(global_counts.round(2).to_string(index=False))

global_counts.to_csv(OUTPUT_DIR / "02_global_class_distribution.csv", index=False)

if "split" in df.columns:
    split_counts = (
        df.groupby(["split", "damage_label"])
        .size()
        .reset_index(name="count")
    )

    split_counts["percentage_within_split"] = (
        split_counts["count"]
        / split_counts.groupby("split")["count"].transform("sum")
        * 100
    )

    split_counts["damage_label"] = pd.Categorical(
        split_counts["damage_label"],
        categories=LABEL_ORDER,
        ordered=True,
    )

    split_counts = split_counts.sort_values(["split", "damage_label"])

    print("\nClass distribution by split:")
    print(split_counts.round(2).to_string(index=False))

    split_counts.to_csv(OUTPUT_DIR / "02_class_distribution_by_split.csv", index=False)

    wide_percentage = split_counts.pivot(
        index="split",
        columns="damage_label",
        values="percentage_within_split",
    ).reset_index()

    wide_percentage.to_csv(
        OUTPUT_DIR / "02_class_distribution_by_split_wide_percentage.csv",
        index=False,
    )

print("\nSaved outputs to:")
print(OUTPUT_DIR)


Global class distribution:
damage_label  count  percentage
   no-damage 196823       73.78
minor-damage  26116        9.79
major-damage  22616        8.48
   destroyed  21225        7.96

Class distribution by split:
split damage_label  count  percentage_within_split
 hold    no-damage  37971                    71.46
 hold minor-damage   6338                    11.93
 hold major-damage   4605                     8.67
 hold    destroyed   4223                     7.95
 test    no-damage  41427                    76.93
 test minor-damage   4798                     8.91
 test major-damage   3850                     7.15
 test    destroyed   3775                     7.01
train    no-damage 117425                    73.49
train minor-damage  14980                     9.37
train major-damage  14161                     8.86
train    destroyed  13227                     8.28

Saved outputs to:
/Users/paolo/Desktop/processed/initial_dataset_exploration


## Dataset analysis

In [3]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.home() / "Desktop"
CSV_PATH = BASE_DIR / "processed" / "buildings_all_with_crops.csv"

OUTPUT_DIR = BASE_DIR / "processed" / "initial_dataset_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)

if "disaster" not in df.columns:
    raise ValueError("Column 'disaster' not found.")

disaster_counts = (
    df.groupby("disaster")
    .size()
    .reset_index(name="building_instances")
    .sort_values("building_instances", ascending=False)
)

disaster_counts["percentage_of_dataset"] = (
    disaster_counts["building_instances"]
    / disaster_counts["building_instances"].sum()
    * 100
)

print("\nBuilding instances by disaster:")
print(disaster_counts.round(2).to_string(index=False))

disaster_counts.to_csv(OUTPUT_DIR / "03_disaster_building_counts.csv", index=False)

if "split" in df.columns:
    disaster_split_counts = (
        df.groupby(["disaster", "split"])
        .size()
        .reset_index(name="building_instances")
        .sort_values(["disaster", "split"])
    )

    print("\nBuilding instances by disaster and split:")
    print(disaster_split_counts.to_string(index=False))

    disaster_split_counts.to_csv(
        OUTPUT_DIR / "03_disaster_building_counts_by_split.csv",
        index=False,
    )

print("\nSaved outputs to:")
print(OUTPUT_DIR)


Building instances by disaster:
           disaster  building_instances  percentage_of_dataset
       palu-tsunami               55178                  20.68
  mexico-earthquake               51362                  19.25
   hurricane-harvey               37374                  14.01
  hurricane-michael               35127                  13.17
  hurricane-matthew               22630                   8.48
santa-rosa-wildfire               21869                   8.20
         socal-fire               18276                   6.85
   midwest-flooding               13423                   5.03
 hurricane-florence               10728                   4.02
  guatemala-volcano                 813                   0.30

Building instances by disaster and split:
           disaster split  building_instances
  guatemala-volcano  hold                 101
  guatemala-volcano  test                  30
  guatemala-volcano train                 682
 hurricane-florence  hold                2491
 

## Disaster analysis

In [4]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.home() / "Desktop"
CSV_PATH = BASE_DIR / "processed" / "buildings_all_with_crops.csv"

OUTPUT_DIR = BASE_DIR / "processed" / "initial_dataset_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)

if "disaster" not in df.columns:
    raise ValueError("Column 'disaster' not found.")

disaster_counts = (
    df.groupby("disaster")
    .size()
    .reset_index(name="building_instances")
    .sort_values("building_instances", ascending=False)
)

disaster_counts["percentage_of_dataset"] = (
    disaster_counts["building_instances"]
    / disaster_counts["building_instances"].sum()
    * 100
)

print("\nBuilding instances by disaster:")
print(disaster_counts.round(2).to_string(index=False))

disaster_counts.to_csv(OUTPUT_DIR / "03_disaster_building_counts.csv", index=False)

if "split" in df.columns:
    disaster_split_counts = (
        df.groupby(["disaster", "split"])
        .size()
        .reset_index(name="building_instances")
        .sort_values(["disaster", "split"])
    )

    print("\nBuilding instances by disaster and split:")
    print(disaster_split_counts.to_string(index=False))

    disaster_split_counts.to_csv(
        OUTPUT_DIR / "03_disaster_building_counts_by_split.csv",
        index=False,
    )

print("\nSaved outputs to:")
print(OUTPUT_DIR)


Building instances by disaster:
           disaster  building_instances  percentage_of_dataset
       palu-tsunami               55178                  20.68
  mexico-earthquake               51362                  19.25
   hurricane-harvey               37374                  14.01
  hurricane-michael               35127                  13.17
  hurricane-matthew               22630                   8.48
santa-rosa-wildfire               21869                   8.20
         socal-fire               18276                   6.85
   midwest-flooding               13423                   5.03
 hurricane-florence               10728                   4.02
  guatemala-volcano                 813                   0.30

Building instances by disaster and split:
           disaster split  building_instances
  guatemala-volcano  hold                 101
  guatemala-volcano  test                  30
  guatemala-volcano train                 682
 hurricane-florence  hold                2491
 

## Disaster damages profiles

In [5]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.home() / "Desktop"
CSV_PATH = BASE_DIR / "processed" / "buildings_all_with_crops.csv"

OUTPUT_DIR = BASE_DIR / "processed" / "initial_dataset_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_ORDER = [
    "no-damage",
    "minor-damage",
    "major-damage",
    "destroyed",
]

df = pd.read_csv(CSV_PATH)

if "disaster" not in df.columns:
    raise ValueError("Column 'disaster' not found.")

df = df[df["damage_label"].isin(LABEL_ORDER)].copy()

counts = (
    df.groupby(["disaster", "damage_label"])
    .size()
    .reset_index(name="count")
)

counts["damage_label"] = pd.Categorical(
    counts["damage_label"],
    categories=LABEL_ORDER,
    ordered=True,
)

counts = counts.sort_values(["disaster", "damage_label"])

percentages = counts.copy()
percentages["percentage_within_disaster"] = (
    percentages["count"]
    / percentages.groupby("disaster")["count"].transform("sum")
    * 100
)

wide_counts = counts.pivot(
    index="disaster",
    columns="damage_label",
    values="count",
).fillna(0).astype(int).reset_index()

wide_percentages = percentages.pivot(
    index="disaster",
    columns="damage_label",
    values="percentage_within_disaster",
).fillna(0).reset_index()

wide_counts["total_buildings"] = wide_counts[LABEL_ORDER].sum(axis=1)

wide_counts = wide_counts.sort_values("total_buildings", ascending=False)

print("\nDamage counts by disaster:")
print(wide_counts.to_string(index=False))

print("\nDamage percentages by disaster:")
print(wide_percentages.round(2).to_string(index=False))

counts.to_csv(OUTPUT_DIR / "04_disaster_damage_counts_long.csv", index=False)
percentages.to_csv(OUTPUT_DIR / "04_disaster_damage_percentages_long.csv", index=False)
wide_counts.to_csv(OUTPUT_DIR / "04_disaster_damage_counts_wide.csv", index=False)
wide_percentages.to_csv(OUTPUT_DIR / "04_disaster_damage_percentages_wide.csv", index=False)

print("\nSaved outputs to:")
print(OUTPUT_DIR)


Damage counts by disaster:
           disaster  no-damage  minor-damage  major-damage  destroyed  total_buildings
       palu-tsunami      46796             1          1178       7203            55178
  mexico-earthquake      51084           221            54          3            51362
   hurricane-harvey      18638          4510         13378        848            37374
  hurricane-michael      22691          8292          2919       1225            35127
  hurricane-matthew       4058         12331          2717       3524            22630
santa-rosa-wildfire      15843           121            95       5810            21869
         socal-fire      15697           136           110       2333            18276
   midwest-flooding      12819           246           193        165            13423
 hurricane-florence       8466           232          1949         81            10728
  guatemala-volcano        731            26            23         33              813

Damage percent

## Image crop diagnostics

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

BASE_DIR = Path.home() / "Desktop"
CSV_PATH = BASE_DIR / "processed" / "buildings_all_with_crops.csv"

OUTPUT_DIR = BASE_DIR / "processed" / "initial_dataset_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_FILES_TO_CHECK = None

df = pd.read_csv(CSV_PATH)

if "crop_path" not in df.columns:
    raise ValueError("Column 'crop_path' not found.")

check_df = df.copy()

if MAX_FILES_TO_CHECK is not None:
    check_df = check_df.sample(
        n=min(MAX_FILES_TO_CHECK, len(check_df)),
        random_state=42,
    )

rows = []

for _, row in tqdm(check_df.iterrows(), total=len(check_df), desc="Checking crops"):
    crop_path = Path(row["crop_path"])

    result = {
        "crop_path": str(crop_path),
        "exists": crop_path.exists(),
        "can_load": False,
        "shape": None,
        "height": None,
        "width": None,
        "channels": None,
        "error": None,
    }

    if crop_path.exists():
        try:
            arr = np.load(crop_path)
            result["can_load"] = True
            result["shape"] = str(arr.shape)

            if arr.ndim == 3:
                result["height"] = arr.shape[0]
                result["width"] = arr.shape[1]
                result["channels"] = arr.shape[2]

        except Exception as e:
            result["error"] = str(e)

    rows.append(result)

diagnostics = pd.DataFrame(rows)

summary = {
    "checked_files": len(diagnostics),
    "missing_files": int((~diagnostics["exists"]).sum()),
    "unloadable_files": int((diagnostics["exists"] & ~diagnostics["can_load"]).sum()),
    "valid_files": int(diagnostics["can_load"].sum()),
    "unique_shapes": diagnostics["shape"].dropna().nunique(),
}

summary_df = pd.DataFrame([summary])

shape_counts = (
    diagnostics["shape"]
    .value_counts(dropna=False)
    .reset_index()
)

shape_counts.columns = ["shape", "count"]

print("\nCrop diagnostics summary:")
print(summary_df.to_string(index=False))

print("\nCrop shape counts:")
print(shape_counts.to_string(index=False))

diagnostics.to_csv(OUTPUT_DIR / "05_crop_diagnostics_full.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "05_crop_diagnostics_summary.csv", index=False)
shape_counts.to_csv(OUTPUT_DIR / "05_crop_shape_counts.csv", index=False)

print("\nSaved outputs to:")
print(OUTPUT_DIR)

Checking crops:   0%|          | 0/266780 [00:00<?, ?it/s]


Crop diagnostics summary:
 checked_files  missing_files  unloadable_files  valid_files  unique_shapes
        266780              0                 0       266780              1

Crop shape counts:
        shape  count
(128, 128, 6) 266780

Saved outputs to:
/Users/paolo/Desktop/processed/initial_dataset_exploration


## Building statistics

In [7]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.home() / "Desktop"
CSV_PATH = BASE_DIR / "processed" / "buildings_all_with_crops.csv"

OUTPUT_DIR = BASE_DIR / "processed" / "initial_dataset_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)

rows = []

for col in ["instance_id", "building_uuid", "building_id", "image_id", "crop_path"]:
    if col in df.columns:
        rows.append({
            "column": col,
            "non_null_count": int(df[col].notna().sum()),
            "unique_count": int(df[col].nunique()),
            "duplicate_count": int(df[col].notna().sum() - df[col].nunique()),
        })

summary_df = pd.DataFrame(rows)

print("\nIdentifier summary:")
print(summary_df.to_string(index=False))

summary_df.to_csv(OUTPUT_DIR / "06_identifier_summary.csv", index=False)

if "image_id" in df.columns:
    buildings_per_image = (
        df.groupby("image_id")
        .size()
        .reset_index(name="building_instances")
        .sort_values("building_instances", ascending=False)
    )

    stats = buildings_per_image["building_instances"].describe().reset_index()
    stats.columns = ["statistic", "value"]

    print("\nBuildings per image summary:")
    print(stats.to_string(index=False))

    buildings_per_image.to_csv(
        OUTPUT_DIR / "06_buildings_per_image.csv",
        index=False,
    )

    stats.to_csv(
        OUTPUT_DIR / "06_buildings_per_image_summary.csv",
        index=False,
    )

if {"bbox_width", "bbox_height"}.issubset(df.columns):
    bbox_df = df[["bbox_width", "bbox_height"]].copy()
    bbox_df["bbox_area"] = bbox_df["bbox_width"] * bbox_df["bbox_height"]

    bbox_summary = bbox_df.describe().reset_index()
    bbox_summary = bbox_summary.rename(columns={"index": "statistic"})

    print("\nBounding box summary:")
    print(bbox_summary.round(2).to_string(index=False))

    bbox_summary.to_csv(
        OUTPUT_DIR / "06_bounding_box_summary.csv",
        index=False,
    )

print("\nSaved outputs to:")
print(OUTPUT_DIR)


Identifier summary:
       column  non_null_count  unique_count  duplicate_count
  instance_id          266780        266780                0
building_uuid          266780        266780                0
     image_id          266780          3731           263049
    crop_path          266780        266780                0

Buildings per image summary:
statistic       value
    count 3731.000000
     mean   71.503618
      std  124.523944
      min    1.000000
      25%    8.000000
      50%   26.000000
      75%   85.000000
      max 1843.000000

Bounding box summary:
statistic  bbox_width  bbox_height  bbox_area
    count   266780.00    266780.00  266780.00
     mean       35.57        34.87    1621.50
      std       24.09        23.27    4097.67
      min        0.92         1.00      16.98
      25%       20.24        19.67     420.00
      50%       31.40        30.69     967.59
      75%       45.01        44.68    1890.27
      max     1022.48       743.10  680656.07

Saved ou

## Final EDA summary tbles

In [8]:
from pathlib import Path
import pandas as pd

# =========================
# Configuration
# =========================

BASE_DIR = Path.home() / "Desktop"
OUTPUT_DIR = BASE_DIR / "processed" / "initial_dataset_exploration"

FINAL_TABLE_DIR = OUTPUT_DIR / "final_summary_tables"
FINAL_TABLE_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Input paths
# =========================

overview_path = OUTPUT_DIR / "01_dataset_overview.csv"
class_path = OUTPUT_DIR / "02_global_class_distribution.csv"
split_class_path = OUTPUT_DIR / "02_class_distribution_by_split.csv"
disaster_path = OUTPUT_DIR / "03_disaster_building_counts.csv"
disaster_damage_path = OUTPUT_DIR / "04_disaster_damage_percentages_wide.csv"
crop_summary_path = OUTPUT_DIR / "05_crop_diagnostics_summary.csv"
shape_counts_path = OUTPUT_DIR / "05_crop_shape_counts.csv"
identifier_path = OUTPUT_DIR / "06_identifier_summary.csv"
building_image_summary_path = OUTPUT_DIR / "06_buildings_per_image_summary.csv"
bbox_summary_path = OUTPUT_DIR / "06_bounding_box_summary.csv"

# =========================
# Helper
# =========================

def display_or_print(df, title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    try:
        from IPython.display import display
        display(df)
    except ImportError:
        print(df.to_string(index=False))


# =========================
# Table 1: General dataset summary
# =========================

summary_rows = []

if overview_path.exists():
    overview = pd.read_csv(overview_path).iloc[0]

    summary_rows.extend([
        {
            "Statistic": "Building instances",
            "Value": int(overview["total_building_instances"]),
        },
        {
            "Statistic": "Columns",
            "Value": int(overview["num_columns"]),
        },
        {
            "Statistic": "Splits",
            "Value": int(overview["num_splits"]),
        },
        {
            "Statistic": "Disasters",
            "Value": int(overview["num_disasters"]),
        },
        {
            "Statistic": "Images",
            "Value": int(overview["num_images"]),
        },
        {
            "Statistic": "Crop files",
            "Value": int(overview["num_crop_paths"]),
        },
    ])

dataset_summary = pd.DataFrame(summary_rows)

dataset_summary.to_csv(
    FINAL_TABLE_DIR / "final_dataset_summary.csv",
    index=False,
)

display_or_print(dataset_summary, "Final Table 1: Dataset summary")

# =========================
# Table 2: Global class distribution
# =========================

if class_path.exists():
    class_df = pd.read_csv(class_path)

    class_summary = class_df.rename(
        columns={
            "damage_label": "Damage label",
            "count": "Building instances",
            "percentage": "Percentage",
        }
    )

    class_summary["Percentage"] = class_summary["Percentage"].round(2)

    class_summary.to_csv(
        FINAL_TABLE_DIR / "final_global_class_distribution.csv",
        index=False,
    )

    display_or_print(class_summary, "Final Table 2: Global class distribution")

# =========================
# Table 3: Class distribution by split
# =========================

if split_class_path.exists():
    split_class = pd.read_csv(split_class_path)

    split_class_wide_counts = split_class.pivot(
        index="split",
        columns="damage_label",
        values="count",
    ).reset_index()

    split_class_wide_pct = split_class.pivot(
        index="split",
        columns="damage_label",
        values="percentage_within_split",
    ).reset_index()

    split_class_wide_counts.columns.name = None
    split_class_wide_pct.columns.name = None

    split_class_wide_counts = split_class_wide_counts.rename(
        columns={"split": "Split"}
    )

    split_class_wide_pct = split_class_wide_pct.rename(
        columns={"split": "Split"}
    )

    for col in split_class_wide_pct.columns:
        if col != "Split":
            split_class_wide_pct[col] = split_class_wide_pct[col].round(2)

    split_class_wide_counts.to_csv(
        FINAL_TABLE_DIR / "final_class_counts_by_split.csv",
        index=False,
    )

    split_class_wide_pct.to_csv(
        FINAL_TABLE_DIR / "final_class_percentages_by_split.csv",
        index=False,
    )

    display_or_print(split_class_wide_counts, "Final Table 3A: Class counts by split")
    display_or_print(split_class_wide_pct, "Final Table 3B: Class percentages by split")

# =========================
# Table 4: Disaster size distribution
# =========================

if disaster_path.exists():
    disaster_df = pd.read_csv(disaster_path)

    disaster_summary = disaster_df.rename(
        columns={
            "disaster": "Disaster",
            "building_instances": "Building instances",
            "percentage_of_dataset": "Percentage of dataset",
        }
    )

    disaster_summary["Percentage of dataset"] = (
        disaster_summary["Percentage of dataset"].round(2)
    )

    disaster_summary.to_csv(
        FINAL_TABLE_DIR / "final_disaster_distribution.csv",
        index=False,
    )

    display_or_print(disaster_summary, "Final Table 4: Disaster distribution")

# =========================
# Table 5: Damage profile by disaster
# =========================

if disaster_damage_path.exists():
    damage_profile = pd.read_csv(disaster_damage_path)

    damage_profile = damage_profile.rename(columns={"disaster": "Disaster"})

    for col in damage_profile.columns:
        if col != "Disaster":
            damage_profile[col] = damage_profile[col].round(2)

    damage_profile.to_csv(
        FINAL_TABLE_DIR / "final_damage_profile_by_disaster.csv",
        index=False,
    )

    display_or_print(damage_profile, "Final Table 5: Damage profile by disaster (%)")

# =========================
# Table 6: Crop diagnostics
# =========================

if crop_summary_path.exists():
    crop_summary = pd.read_csv(crop_summary_path)

    crop_summary = crop_summary.rename(
        columns={
            "checked_files": "Checked crop files",
            "missing_files": "Missing files",
            "unloadable_files": "Unloadable files",
            "valid_files": "Valid files",
            "unique_shapes": "Unique crop shapes",
        }
    )

    crop_summary.to_csv(
        FINAL_TABLE_DIR / "final_crop_diagnostics_summary.csv",
        index=False,
    )

    display_or_print(crop_summary, "Final Table 6: Crop diagnostics summary")

if shape_counts_path.exists():
    shape_counts = pd.read_csv(shape_counts_path)

    shape_counts = shape_counts.rename(
        columns={
            "shape": "Crop shape",
            "count": "Count",
        }
    )

    shape_counts.to_csv(
        FINAL_TABLE_DIR / "final_crop_shape_counts.csv",
        index=False,
    )

    display_or_print(shape_counts, "Final Table 7: Crop shape counts")

# =========================
# Table 7: Identifier and image statistics
# =========================

if identifier_path.exists():
    identifier_df = pd.read_csv(identifier_path)

    identifier_summary = identifier_df.rename(
        columns={
            "column": "Identifier",
            "non_null_count": "Non-null count",
            "unique_count": "Unique count",
            "duplicate_count": "Duplicate count",
        }
    )

    identifier_summary.to_csv(
        FINAL_TABLE_DIR / "final_identifier_summary.csv",
        index=False,
    )

    display_or_print(identifier_summary, "Final Table 8: Identifier summary")

if building_image_summary_path.exists():
    image_stats = pd.read_csv(building_image_summary_path)

    image_stats = image_stats.rename(
        columns={
            "statistic": "Statistic",
            "value": "Buildings per image",
        }
    )

    image_stats["Buildings per image"] = image_stats["Buildings per image"].round(2)

    image_stats.to_csv(
        FINAL_TABLE_DIR / "final_buildings_per_image_summary.csv",
        index=False,
    )

    display_or_print(image_stats, "Final Table 9: Buildings per image summary")

# =========================
# Table 8: Bounding box statistics
# =========================

if bbox_summary_path.exists():
    bbox_summary = pd.read_csv(bbox_summary_path)

    bbox_summary = bbox_summary.rename(
        columns={
            "statistic": "Statistic",
            "bbox_width": "Bounding box width",
            "bbox_height": "Bounding box height",
            "bbox_area": "Bounding box area",
        }
    )

    for col in bbox_summary.columns:
        if col != "Statistic":
            bbox_summary[col] = bbox_summary[col].round(2)

    bbox_summary.to_csv(
        FINAL_TABLE_DIR / "final_bounding_box_summary.csv",
        index=False,
    )

    display_or_print(bbox_summary, "Final Table 10: Bounding box summary")

print("\nSaved final summary tables to:")
print(FINAL_TABLE_DIR)


Final Table 1: Dataset summary


,Statistic,Value
0,Building instances,266780
1,Columns,18
2,Splits,3
3,Disasters,10
4,Images,3731
5,Crop files,266780



Final Table 2: Global class distribution


,Damage label,Building instances,Percentage
0,no-damage,196823,73.78
1,minor-damage,26116,9.79
2,major-damage,22616,8.48
3,destroyed,21225,7.96



Final Table 3A: Class counts by split


...


Final Table 9: Buildings per image summary


,Statistic,Buildings per image
0,count,3731.00
1,mean,71.50
2,std,124.52
3,min,1.00
4,25%,8.00
5,50%,26.00
6,75%,85.00
7,max,1843.00



Final Table 10: Bounding box summary


,Statistic,Bounding box width,Bounding box height,Bounding box area
0,count,266780.00,266780.00,266780.00
1,mean,35.57,34.87,1621.50
2,std,24.09,23.27,4097.67
3,min,0.92,1.00,16.98
4,25%,20.24,19.67,420.00
5,50%,31.40,30.69,967.59
6,75%,45.01,44.68,1890.27
7,max,1022.48,743.10,680656.07



Saved final summary tables to:
/Users/paolo/Desktop/processed/initial_dataset_exploration/final_summary_tables
